# Meta Co-Training on CIFAR-10 (free-tier Colab)

This notebook fine-tunes the **Meta Co-Training (MCT)** method from [arXiv:2311.18083](https://arxiv.org/abs/2311.18083) on a small dataset, sized to fit a free Colab GPU (T4, single device).

It is self-contained: running it top to bottom clones the original paper's code, writes a CIFAR-10 training script into it, and runs it. You only need to upload *this notebook* to your own repo.

## What is Meta Co-Training?
- **Co-training** is a semi-supervised learning idea: train two models on two different "views" of the same data. Each model generates pseudo-labels for unlabeled examples, which are used to teach the other model. This works when the two views are *independent* and each is *sufficient* on its own to solve the task.
- Getting genuinely independent views is hard in practice. This paper's trick: use two different **pretrained (frozen) foundation models** as the two views (e.g. CLIP and DINOv2), since they were trained with different objectives/data and so encode different, complementary information about the same image.
- Plain co-training on these constructed views works but has some failure modes (e.g. the two models can drift/collapse together). **Meta Co-Training** fixes this by borrowing from **Meta Pseudo Labels**: instead of just training on the other model's pseudo-labels directly, each model's update is adjusted based on how much that pseudo-label actually *helped* the other model on a small held-out labeled batch (a meta-learning / bi-level optimization step). This makes the pseudo-label signal self-correcting instead of being treated as ground truth.
- Result: new SOTA on ImageNet-10% with very little compute, and strong results on fine-grained classification datasets.

## Why CIFAR-10
- No manual download link needed — `torchvision.datasets.CIFAR10` fetches it automatically (~170MB) from the dataset's own host.
- 60,000 32x32 images, 10 classes: small enough to iterate fast on a free Colab GPU.
- Caveat: CIFAR's images are tiny (32x32) and get upsampled to 224x224 for the CLIP/DINOv2 backbones, so they'll look blurry to those models — this trades realism for speed, which is the right tradeoff for a free-tier deliverable. If you want sharper fine-grained results later, Oxford-IIIT Pet is a better (slower) choice.

## Why this setup is different from the paper's default usage
- The repo is built around **one GPU per view**, launched with `torchrun`. Free Colab gives you exactly **one** GPU.
- The good news: the code picks its device with `rank % num_gpus`, so if there's only 1 GPU, every process just uses GPU 0. We can still launch 2 processes (one per view: DINOv2 and CLIP) with `torchrun --standalone --nproc_per_node=2` and they'll share the single GPU. This is slower than true multi-GPU, but it is the actual MCT algorithm, not an approximation of it.
- Both foundation models stay **frozen** (only a small MLP head is trained on top), so this is cheap enough for a T4.

We keep `--train_size` fraction of the training set as labeled data and treat the rest as unlabeled — exactly the kind of low-label regime MCT targets.

## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Clone the original Meta Co-Training implementation

This gets us the `mct` package (the actual MCT algorithm: `mct/MCT.py`, the CLIP/DINOv2 wrappers in `mct/image_models.py`, etc). We then add our own CIFAR-10 training script into its `examples/` folder in the next cell.

In [ ]:
!git clone https://github.com/JayRothenberger/Meta-Co-Training.git /content/Meta-Co-Training
%cd /content/Meta-Co-Training


## 3. Install dependencies

In [ ]:
!pip install -e meta-cotraining
!pip install git+https://github.com/openai/CLIP.git
!pip install transformers timm scikit-image scipy wandb


## 4. Write the CIFAR-10 training script

Adapts the repo's `examples/MCT_e2e.py` (which targets full ImageNet on a multi-GPU cluster) to a 10-class, single-GPU-friendly run: CIFAR-10 instead of ImageNet, a generic stratified percent-split, and wandb disabled by default (no API key needed).

In [ ]:
%%writefile /content/Meta-Co-Training/examples/MCT_e2e_cifar.py
# standard library imports
import argparse
import os
from copy import deepcopy as copy

# installed imports
import numpy as np
import torch
import torchvision

# local code imports
from mct.MCT import MetaCoTrainingModel
from mct.image_models import IMAGE_DISTANCES, IMAGE_TRANSFORMS
from mct.models import MLPProbe

NUM_CLASSES = 10  # CIFAR-10
VIEWS = ['DINOv2', 'CLIP']


def get_labels(dataset):
    if hasattr(dataset, '_labels'):
        return list(dataset._labels)
    if hasattr(dataset, 'targets'):
        return list(dataset.targets)
    return [dataset[i][1] for i in range(len(dataset))]


def percent_split_indices(labels, percent, seed=13):
    """stratified split: `percent` (0-1) of each class goes to the labeled set, the rest unlabeled"""
    np.random.seed(seed)
    labels = np.array(labels)
    train_idx = []
    for c in np.unique(labels):
        idx_c = np.where(labels == c)[0]
        n_keep = max(1, int(round(len(idx_c) * percent)))
        train_idx.extend(np.random.choice(idx_c, n_keep, replace=False).tolist())
    train_idx = set(train_idx)
    unlbl_idx = [i for i in range(len(labels)) if i not in train_idx]
    return sorted(train_idx), unlbl_idx


def training_process(args):
    device = int(os.environ['RANK']) % torch.cuda.device_count()

    torch.manual_seed(13)
    torch.cuda.set_device(device)

    rank = int(os.environ['RANK'])

    trains, unlbls, vals = [], [], []

    # each view gets its own wandb process; wandb disabled by default so this works with no API key
    if rank < len(VIEWS):
        import wandb
        wandb.init(project='MCT CIFAR-10', mode='disabled' if not args.wandb else 'online',
                   name=f"{rank}: {args.train_size}", config={'args': vars(args)})

    full_labels = None
    train_idx = unlbl_idx = None

    for view in VIEWS:
        trainval = torchvision.datasets.CIFAR10(
            args.dataset_path, train=True, transform=IMAGE_TRANSFORMS[view], download=True)
        test = torchvision.datasets.CIFAR10(
            args.dataset_path, train=False, transform=IMAGE_TRANSFORMS[view], download=True)

        if full_labels is None:
            full_labels = get_labels(trainval)
            train_idx, unlbl_idx = percent_split_indices(full_labels, args.train_size)

        trains.append(torch.utils.data.Subset(trainval, train_idx))
        unlbls.append(torch.utils.data.Subset(trainval, unlbl_idx))
        vals.append(test)

    models = [IMAGE_DISTANCES[view]().model.to(device) for view in VIEWS]

    shapes = []
    with torch.no_grad():
        for train, model in zip(trains, models):
            x0 = train[0][0].unsqueeze(0).to(device)
            shapes.append(model(x0).shape[-1])

    models = [MLPProbe(model, shape, NUM_CLASSES) for shape, model in zip(shapes, models)]

    os.makedirs('./chkpts', exist_ok=True)

    # supervised warmup of the MLP heads on the frozen foundation-model features
    MCTModel = MetaCoTrainingModel(models)
    states = MCTModel.train(
        args.warmup_epochs, args.warmup_epochs + 1,
        copy(trains), copy(unlbls), copy(vals), copy(vals),
        checkpoint_path='./chkpts/cifar_warmup', batch_size=args.batch_size,
        lr=args.learning_rate, log_interval=args.log_interval, amp=True,
    )

    # meta co-training stage: each view's model pseudo-labels the other's unlabeled data
    MCTModel = MetaCoTrainingModel(models)
    torch.distributed.barrier()
    states = MCTModel.train(
        args.epochs, 0,
        copy(trains), copy(unlbls), copy(vals), copy(vals),
        checkpoint_path='./chkpts/cifar_mct', batch_size=args.batch_size,
        lr=args.learning_rate, log_interval=args.log_interval,
        approx=True, amp=True,
    )

    return states


def setup(rank, world_size):
    torch.distributed.init_process_group("nccl", rank=rank, world_size=world_size)


def cleanup():
    torch.distributed.destroy_process_group()


def main(args, rank, world_size):
    setup(rank, world_size)

    states = training_process(args)

    if rank == 0:
        print('saving checkpoint')
        torch.save(states, './chkpts/cifar_final_states.pt')

    print('cleanup')
    cleanup()


def create_parser():
    parser = argparse.ArgumentParser(description='Meta Co-Training on CIFAR-10 (Colab-friendly, 2 views)')

    parser.add_argument('--warmup_epochs', type=int, default=3,
                        help='supervised warmup epochs before meta co-training (default: 3)')
    parser.add_argument('--epochs', type=int, default=5,
                        help='meta co-training epochs (default: 5)')
    parser.add_argument('-b', '--batch_size', type=int, default=32,
                        help='batch size for training (default: 32)')
    parser.add_argument('-lr', '--learning_rate', type=float, default=1e-3,
                        help='learning rate (default: 1e-3)')
    parser.add_argument('--dataset_path', type=str, default='./data',
                        help='directory to download/read CIFAR-10 into')
    parser.add_argument('--train_size', type=float, default=0.1,
                        help='fraction (0-1) of data kept as the labeled training set, rest is unlabeled (default: 0.1)')
    parser.add_argument('--log_interval', type=int, default=20,
                        help='steps between validation/checkpoint logging (default: 20)')
    parser.add_argument('--wandb', action='store_true',
                        help='log to wandb instead of running with wandb disabled')

    return parser


if __name__ == '__main__':
    parser = create_parser()
    args = parser.parse_args()

    world_size = int(os.environ["WORLD_SIZE"])
    rank = int(os.environ["RANK"])
    torch.multiprocessing.set_start_method('spawn')

    main(args, rank, world_size)


## 5. Write the launcher

Runs 2 processes (one per view) on the single Colab GPU via `torchrun --standalone`. Works because `MCT_e2e_cifar.py` picks its device as `rank % num_gpus` — with 1 GPU, both ranks land on GPU 0.

In [ ]:
%%writefile /content/Meta-Co-Training/examples/mct_e2e_cifar.sh
#!/bin/bash
# Colab-friendly launcher: runs 2 processes (one per view) on a single GPU via
# torchrun's --standalone mode. No SLURM/cluster required.
set -e

cd "$(dirname "$0")"

torchrun --standalone --nproc_per_node=2 MCT_e2e_cifar.py \
  --dataset_path "${DATASET_PATH:-./data}" \
  --warmup_epochs "${WARMUP_EPOCHS:-3}" \
  --epochs "${EPOCHS:-5}" \
  --batch_size "${BATCH_SIZE:-32}" \
  --train_size "${TRAIN_SIZE:-0.1}" \
  --log_interval "${LOG_INTERVAL:-20}"


## 6. Smoke test (tiny run, ~1-2 min)
Confirms the pipeline (data download, both foundation models, the MCT training loop) runs end-to-end before committing GPU time to a full run.

In [ ]:
%cd /content/Meta-Co-Training/examples
%env DATASET_PATH=/content/data
%env WARMUP_EPOCHS=1
%env EPOCHS=1
%env BATCH_SIZE=16
%env TRAIN_SIZE=0.1
%env LOG_INTERVAL=5
!bash mct_e2e_cifar.sh


## 7. Full fine-tuning run
Bump the epoch counts once the smoke test passes. Checkpoints land in `examples/chkpts/`.

In [ ]:
%cd /content/Meta-Co-Training/examples
%env DATASET_PATH=/content/data
%env WARMUP_EPOCHS=5
%env EPOCHS=10
%env BATCH_SIZE=32
%env TRAIN_SIZE=0.1
%env LOG_INTERVAL=20
!bash mct_e2e_cifar.sh


## 8. Reading the output
- `val_acc0` / `val_acc1`: validation accuracy of each individual view's model (DINOv2-head, CLIP-head).
- `c_acc`: **co-accuracy**, the combined prediction of both views together (sum of logits) — this is MCT's headline number, and should end up higher than either view alone if co-training is working.
- `test_acc0` / `test_acc1` / `c_acc_test`: the same metrics on the held-out test split.
- Checkpoints: `examples/chkpts/cifar_warmup_*` (after supervised warmup) and `examples/chkpts/cifar_mct_*` (after meta co-training).